# CNN-LSTM experiment
This notebook follows `training_gru.ipynb`, replacing only the GRU with a unidirectional LSTM. Run from the repository root or `main_notebooks/`.

Two one-layer experiments: `LSTM_HIDDEN_SIZE = 64` matches the GRU's hidden width; `55` approximately matches the total parameter count of the default GRU-64 model. Keep data, label delay, augmentation, seed, optimizer, and stopping settings identical for a 1-v-1 comparison.

The LSTM uses `batch_first=True`, `bias=True`, no bidirectionality or projection, and PyTorch's default initialization. Dropout remains before the recurrent layer and in the head, as in the GRU baseline; recurrent inter-layer dropout is zero. Both hidden and cell states reset at each training window.

Checkpoints include `model_type='lstm'`, the LSTM configuration, target delay, and amplitude-augmentation setting. The evaluation notebooks and comparison example loader support this model type.

# Imports

In [18]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import random
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from datetime import datetime

import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Start from the repo root or main_notebooks/'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks'))

from tap_recognition.config import DataConfig, LSTMModelConfig, \
    TrainConfig, TrainingConfig, LabelConfig, EarlyStoppingConfig
from tap_recognition.dataset import RecordedIMUDataset
from tap_recognition.model_lstm import CausalCNNLSTM
from train import build_datasets, EarlyStopping


In [19]:
LSTM_HIDDEN_SIZE = 64  # Use 55 for an approximately parameter-matched GRU-64 comparison.
LSTM_LAYERS = 1
LABEL_DELAY_FRAMES = 8  # Use 8 when comparing against the delayed-8 GRU experiment.
AMPLITUDE_AUGMENTATION = True  # Match the GRU run being compared.

cfg = TrainConfig(
    seed=62,
    out_dir='testing_checkpoints',
    data=DataConfig(train_dir='data_09_10_26/train_data', val_dir='data_09_10_26/valid_data', highpass=True),
    model=LSTMModelConfig(lstm_hidden=LSTM_HIDDEN_SIZE, lstm_layers=LSTM_LAYERS, ),
    training=TrainingConfig(early_stopping=EarlyStoppingConfig(), epochs=60),
)

for key in ('train_dir', 'val_dir'):
    setattr(cfg.data, key, ','.join(str(ROOT / p.strip()) for p in getattr(cfg.data, key).split(',')))

print(json.dumps(cfg.to_dict(), indent=2))


{
  "seed": 62,
  "device": "auto",
  "out_dir": "testing_checkpoints",
  "init_checkpoint": null,
  "freeze_except_last": false,
  "data": {
    "mode": "recorded",
    "train_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/train_data",
    "val_dir": "/home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/valid_data",
    "window_samples": 300,
    "sample_rate": 100.0,
    "highpass": true,
    "negative_windows_per_file": 20,
    "trigger_context_samples": 200,
    "exclusion_margin": 200,
    "dt_min": 0.12,
    "dt_max": 0.45,
    "session_exclusions_file": "data/session_exclusions.csv",
    "synthetic_samples": 8000,
    "val_split": 0.15
  },
  "labels": {
    "sigma": 0.04,
    "peak_offset": 0.05,
    "half_frames": 10
  },
  "model": {
    "input_dim": 6,
    "num_classes": 3,
    "cnn_channels": 32,
    "lstm_hidden": 64,
    "lstm_layers": 1,
    "kernel_size": 5,
    "dilations":

# Seed setting

In [20]:
def fix_seeds(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed=seed)
    random.seed(seed)
    torch.use_deterministic_algorithms(True)
fix_seeds(cfg.seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Train and Validation Datasets

In [21]:
train_ds, val_ds = build_datasets(cfg)
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=cfg.training.batch_size,
    shuffle=True,
    num_workers=cfg.training.num_workers
)
val_loader = DataLoader(
    dataset=val_ds,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    num_workers=cfg.training.num_workers
)

Train dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/train_data
Val dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_09_10_26/valid_data


# Model creation

In [22]:
model_kwargs = cfg.model.to_model_kwargs()
model_kwargs

{'input_dim': 6,
 'num_classes': 3,
 'cnn_channels': 32,
 'lstm_hidden': 64,
 'lstm_layers': 1,
 'kernel_size': 5,
 'dilations': (1, 2, 4),
 'dropout': 0.1}

In [23]:
assert (cfg.init_checkpoint is None), 'We are testing models training from scratch'

In [24]:
model = CausalCNNLSTM(**model_kwargs).to(device)
assert not cfg.freeze_except_last, 'No fine tuning'
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = ROOT / Path(cfg.out_dir) / run_timestamp
out_dir.mkdir(parents=True, exist_ok=True)


Model parameters: 43,139


In [25]:
trainable = list(model.parameters())

# Optimizer and Learning Rate

In [26]:
optimizer = torch.optim.AdamW(
    trainable,
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.training.epochs,
)

# Early Stopping and train loop configuration

In [27]:
best_val_acc = 0.0
early_stop = EarlyStopping(cfg.training.early_stopping)
stopped_epoch = cfg.training.epochs

In [28]:
cfg_es = cfg.training.early_stopping

## Positive and negative counts

In [29]:
train_ds[0]

{'imu': tensor([[-6.5455e-01,  5.9202e+00,  6.2915e+00,  1.7679e-01, -3.0184e-01,
          -2.8750e-02],
         [-5.9321e-01,  5.6899e+00,  5.8555e+00,  1.4784e-01, -3.2540e-01,
          -5.4712e-03],
         [-5.0817e-01,  5.5149e+00,  5.4686e+00,  8.4440e-02, -3.1576e-01,
           1.5015e-02],
         ...,
         [ 2.0657e-02,  4.6460e-02,  5.2779e-02, -1.8321e-02,  3.8827e-02,
           1.3318e-02],
         [ 5.0951e-03,  4.0525e-02,  3.7283e-02,  3.8144e-04,  1.9681e-02,
           1.1943e-02],
         [ 2.7383e-02,  3.9193e-02,  1.8024e-02,  1.9742e-02, -6.9042e-03,
           1.0518e-02]]),
 'frame_labels': tensor([[1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000],
         [1.0000,

In [30]:
n_train_pos = 0
for i in range(len(train_ds)):
    # imu = train_ds[i]['imu']
    # label = train_ds[i]['frame_labels']
    window = train_ds[i]['window_label']
    if window > 0:
        n_train_pos += 1

n_val_pos = 0
for i in range(len(val_ds)):
    window = val_ds[i]['window_label']
    if window > 0:
        n_val_pos += 1

print(
    f"Training on {device}, {len(train_ds)} train "
    f"({n_train_pos} pos / {len(train_ds) - n_train_pos} neg) / "
    f"{len(val_ds)} val ({n_val_pos} pos / {len(val_ds) - n_val_pos} neg)"
)

print(
    f"Sequence length: {cfg.data.window_samples} frames "
    f"({cfg.data.window_samples / cfg.data.sample_rate:.2f} s)"
)
print(f"Model receptive field: {model.receptive_field} samples")
if cfg_es.enabled:
    print(
        f"Early stopping: monitor={cfg_es.monitor}, patience={cfg_es.patience}, "
        f"min_delta={cfg_es.min_delta}"
    )

Training on cpu, 2532 train (1412 pos / 1120 neg) / 1018 val (578 pos / 440 neg)
Sequence length: 300 frames (3.00 s)
Model receptive field: 29 samples
Early stopping: monitor=val_window_acc, patience=10, min_delta=0.001


# Train loop
Training and validation use the same delayed soft frame targets for loss and frame accuracy. The original `window_label` and detected event frame remain unchanged, so window-level metrics and timing references still refer to the original annotation.

In [31]:
from train import _augment_positive_windows, frame_soft_ce, window_class_from_probs

# Shift only the generated soft targets, not the event-frame annotations.
def shift_event_targets_right(frame_labels, shift):
    """Move event probabilities later and assign remaining mass to none."""
    if shift < 0:
        raise ValueError("shift must be non-negative")
    if shift == 0:
        return frame_labels
    if shift >= frame_labels.shape[1]:
        raise ValueError("shift must be smaller than the number of frames")

    shifted = torch.zeros_like(frame_labels)
    shifted[..., 0] = 1.0
    shifted[:, shift:, 1:] = frame_labels[:, :-shift, 1:]
    shifted[..., 0] = 1.0 - shifted[..., 1:].sum(dim=-1)
    return shifted

## Evaluation

In [32]:
@torch.no_grad()
def evaluate(
    model: CausalCNNLSTM,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
    event_weight: float = 10.0,
    neg_window_weight: float = 2.0,
) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    correct_frames = 0
    total_frames = 0
    correct_windows = 0
    total_windows = 0
    tp = fp = fn = tn = 0

    for batch in loader:
        imu = batch["imu"].to(device)
        frame_labels = batch["frame_labels"].to(device)
        frame_labels = shift_event_targets_right(frame_labels, LABEL_DELAY_FRAMES)
        window_labels = batch["window_label"].to(device)

        logits, _ = model(imu)
        probs = torch.softmax(logits, dim=-1)
        loss = frame_soft_ce(
            logits,
            frame_labels,
            event_weight,
            window_label=window_labels,
            neg_window_weight=neg_window_weight,
        )
        total_loss += loss.item() * imu.size(0)

        pred = probs.argmax(dim=-1)
        true_cls = frame_labels.argmax(dim=-1)
        correct_frames += (pred == true_cls).sum().item()
        total_frames += true_cls.numel()

        window_pred = window_class_from_probs(probs, threshold)
        correct_windows += (window_pred == window_labels).sum().item()
        total_windows += imu.size(0)

        pos_pred = (window_pred > 0).long()
        pos_true = (window_labels > 0).long()
        for wp, wl in zip(pos_pred, pos_true):
            if wp == 1 and wl == 1:
                tp += 1
            elif wp == 1 and wl == 0:
                fp += 1
            elif wp == 0 and wl == 1:
                fn += 1
            else:
                tn += 1

    n = len(loader.dataset)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return {
        "loss": total_loss / n,
        "frame_acc": correct_frames / total_frames,
        "window_acc": correct_windows / total_windows,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

In [33]:
def train_one_epoch(
    model: CausalCNNLSTM,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    grad_clip: float,
    threshold: float,
    event_weight: float,
    freeze_except_last: bool = False,
    neg_window_weight: float = 2.0,
) -> dict[str, float]:
    if freeze_except_last:
        model.eval()
    else:
        model.train()
    total_loss = 0.0
    correct_frames = 0
    total_frames = 0
    correct_windows = 0
    total_windows = 0

    for batch in loader:
        imu = batch["imu"].to(device)
        frame_labels = batch["frame_labels"].to(device)
        frame_labels = shift_event_targets_right(frame_labels, LABEL_DELAY_FRAMES)
        window_label = batch["window_label"].to(device)
        if AMPLITUDE_AUGMENTATION:
            imu = _augment_positive_windows(imu, window_label)

        logits, _ = model(imu)
        loss = frame_soft_ce(
            logits,
            frame_labels,
            event_weight,
            window_label=window_label,
            neg_window_weight=neg_window_weight,
        )
        probs = torch.softmax(logits, dim=-1)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item() * imu.size(0)
        pred = probs.argmax(dim=-1)
        true_cls = frame_labels.argmax(dim=-1)
        correct_frames += (pred == true_cls).sum().item()
        total_frames += true_cls.numel()

        window_pred = window_class_from_probs(probs, threshold)
        window_true = batch["window_label"].to(device)
        correct_windows += (window_pred == window_true).sum().item()
        total_windows += imu.size(0)

    n = len(loader.dataset)
    return {
        "loss": total_loss / n,
        "frame_acc": correct_frames / total_frames,
        "window_acc": correct_windows / total_windows,
    }

In [34]:
history = []

for epoch in range(1, cfg.training.epochs + 1):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device,
        cfg.training.grad_clip,
        cfg.training.prediction_threshold,
        cfg.training.event_loss_weight,
        freeze_except_last=cfg.freeze_except_last,
        neg_window_weight=cfg.training.neg_window_weight,
    )
    val_metrics = evaluate(
        model,
        val_loader,
        device,
        cfg.training.prediction_threshold,
        cfg.training.event_loss_weight,
        neg_window_weight=cfg.training.neg_window_weight,
    )

    scheduler.step()

    print(
        f"Epoch {epoch:3d}/{cfg.training.epochs} | "
        f"train loss={train_metrics['loss']:.4f} acc={train_metrics['window_acc']:.3f} | "
        f"val loss={val_metrics['loss']:.4f} acc={val_metrics['window_acc']:.3f} "
        f"P={val_metrics['precision']:.3f} R={val_metrics['recall']:.3f}"
    )

    if val_metrics["window_acc"] > best_val_acc:
        best_val_acc = val_metrics["window_acc"]
        torch.save(
            {
                "model_type": "lstm",
                "model_state": model.state_dict(),
                "model_config": model_kwargs,
                "train_config": cfg.to_dict(),
                "positive_label_delay_frames": LABEL_DELAY_FRAMES,
                "amplitude_augmentation": AMPLITUDE_AUGMENTATION,
                "epoch": epoch,
                "val_metrics": val_metrics,
            },
            out_dir / "best.pt",
        )

    history.append({
        "epoch": epoch,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
    })

    if early_stop.step(val_metrics):
        stopped_epoch = epoch
        print(
            f"Early stopping at epoch {epoch} "
            f"(no {cfg_es.monitor} improvement for {cfg_es.patience} epochs)"
        )
        break

torch.save(
    {
        "model_type": "lstm",
        "model_state": model.state_dict(),
        "model_config": model_kwargs,
        "train_config": cfg.to_dict(),
        "positive_label_delay_frames": LABEL_DELAY_FRAMES,
        "amplitude_augmentation": AMPLITUDE_AUGMENTATION,
        "epoch": stopped_epoch,
    },
    out_dir / "last.pt",
)
print(f"Done. Best val window acc: {best_val_acc:.3f} (stopped at epoch {stopped_epoch})")
print(f"Checkpoints saved to {out_dir}/")

pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)


Epoch   1/60 | train loss=0.5667 acc=0.444 | val loss=0.4090 acc=0.439 P=1.000 R=0.012
Epoch   2/60 | train loss=0.3864 acc=0.555 | val loss=0.3437 acc=0.648 P=0.956 R=0.413
Epoch   3/60 | train loss=0.2924 acc=0.809 | val loss=0.3173 acc=0.838 P=0.935 R=0.770
Epoch   4/60 | train loss=0.2569 acc=0.861 | val loss=0.3077 acc=0.863 P=0.942 R=0.815
Epoch   5/60 | train loss=0.2354 acc=0.902 | val loss=0.2325 acc=0.929 P=0.941 R=0.938
Epoch   6/60 | train loss=0.2243 acc=0.915 | val loss=0.2582 acc=0.883 P=0.899 R=0.912
Epoch   7/60 | train loss=0.2268 acc=0.908 | val loss=0.2273 acc=0.935 P=0.940 R=0.948
Epoch   8/60 | train loss=0.2182 acc=0.921 | val loss=0.2222 acc=0.945 P=0.952 R=0.953
Epoch   9/60 | train loss=0.2039 acc=0.933 | val loss=0.2621 acc=0.927 P=0.974 R=0.898
Epoch  10/60 | train loss=0.2013 acc=0.944 | val loss=0.2628 acc=0.920 P=0.972 R=0.888
Epoch  11/60 | train loss=0.1982 acc=0.945 | val loss=0.2296 acc=0.957 P=0.973 R=0.952
Epoch  12/60 | train loss=0.1976 acc=0.948 